<a href="https://colab.research.google.com/github/chantakorn-uranukul-5792092/chantakorn-uranukul-5792092-AnalyticsEngineering/blob/main/5_02_practice_weekly_ops_attempt1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=17M9oGaOj4KdKiDDH3oTibxJ9qBTSjp6n)

# 5.02 Analytics Engineering: practice test
## Practice: Weekly operations pipeline

## How this practice works

This is a rehearsal for the in-class test. It is **not marked** and nothing is submitted. It works only if you treat it like the real thing: attempt it first, and only then look at the model answer.

This is set to be the same time given in the real test; the same style of notebook and problem; and as close to the real thing as possible. However, of course, the problem is different; the data is different; and you will do it asychronously (which is also different).

- Set a timer for **55 minutes** and work alone. Stop when it rings, then spend **5 minutes** exporting your notebook (see below). Write your start and stop times in the cell under the timing table.
- Work in **Google Colab** and save the notebook as `.ipynb`.
- You may use *any materials, the module GitHub code and any generative AI tool*.
- There is no end-point as such. The task is longer than 55 minutes, so you are completing part of a larger job. Do as much as you can, and do it properly.
- The code itself is not marked. Marks in the real test go on how the task was executed, the methodology applied, and the interpretation of results. Practise that.
- In text cells or comments, include:
  1. **how the code was generated**: the prompt you sent, or the source you used;
  2. **evidence of checking**: what you compared, counted or inspected, and what it showed;
  3. **your interpretation in your own words**.

**Tips**

- Run small pieces and know what output you expect before you run them.
- Put enough text next to the code that someone else can follow your reasoning.
- Follow sound analytics engineering method: understand the grain/rows that you are building around, clean in stages, check before you move on.
- Quality over quantity. A smaller pipeline that you have checked beats a larger one you have not.
- Stay calm. Not finishing is expected.

**Suggested timing**

| Minutes | Focus |
| --- | --- |
| 0 to 10 | Explore and profile the tables |
| 10 to 20 | Plan: grain, joins, cleaning decisions |
| 20 to 45 | Build as far as you can |
| 45 to 55 | Check, reconcile and write up |
| 55 to 60 | Export: **Runtime, then Restart session and run all**; File, Download, .ipynb; put the file in a zip |

In the real test you upload the zip to the Class Test area on my.wbs. Here, keep it so you can compare your work with the model answer afterwards.

**Started at:** ____  **Stopped at:** ____  **Furthest step I reached (see the brief):** ____

## The data

You have six tables from a fictional food-delivery marketplace, Plateful, which works with restaurants in five UK cities. They cover orders placed from 1 March to 30 June 2026.

| Name in this notebook | What it holds |
| --- | --- |
| `orders` | One record per order: customer, restaurant, when it was placed, delivery fee |
| `order_status_events` | The order's progress through the process, recorded as events |
| `order_lines` | The items inside each order |
| `payments` | Money movements for each order: charges and refunds |
| `restaurants` | Restaurant details: cuisine and city |
| `customers` | Customer details |

All timestamps in all tables are stored in **UTC**.

**How the tables connect**

- `orders` to `order_status_events`, `order_lines` and `payments` on `order_id`
- `orders` to `restaurants` on `restaurant_id`
- `orders` to `customers` on `customer_id`

Check the actual column names in the next cell before you write any joins.

In [ ]:
# ---- Setup: run this cell first ----
!pip install -q duckdb

import glob
import os

import duckdb
import pandas as pd

DATA_URL = "https://github.com/MJMortensonWarwick/adminaeaiv/releases/download/practicesession/plateful-practice.zip"    # link to plateful-practice.zip (the CSV files are inside a folder in the zip)

DATA_DIR = "/content/plateful"

if DATA_URL and not glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True):
    import urllib.request
    import zipfile
    os.makedirs(DATA_DIR, exist_ok=True)
    urllib.request.urlretrieve(DATA_URL, "/content/plateful.zip")
    zipfile.ZipFile("/content/plateful.zip").extractall(DATA_DIR)

FILES = {
    "orders": "orders.csv",
    "order_status_events": "order_status_events.csv",
    "order_lines": "order_lines.csv",
    "payments": "payments.csv",
    "restaurants": "restaurants.csv",
    "customers": "customers.csv",
}

tables, missing = {}, []
for name, filename in FILES.items():
    found = glob.glob(os.path.join(DATA_DIR, "**", filename), recursive=True)
    if found:
        tables[name] = pd.read_csv(found[0])
    else:
        missing.append(filename)

if missing:
    found_csvs = sorted(os.path.basename(f) for f in glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True))
    raise FileNotFoundError(f"Missing files: {missing}. CSV files found under {DATA_DIR}: {found_csvs}.")

# Make every table available as a DuckDB table, so you can use SQL or pandas.
con = duckdb.connect()
for name, df in tables.items():
    con.register(name, df)


def q(sql):
    """Run SQL against the tables above and return a pandas DataFrame."""
    return con.sql(sql).df()


for name, df in tables.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")
    print("   ", list(df.columns))

orders: 14,494 rows, 5 columns
    ['order_id', 'customer_id', 'restaurant_id', 'order_ts', 'delivery_fee']
order_status_events: 56,812 rows, 4 columns
    ['event_id', 'order_id', 'status', 'event_ts']
order_lines: 36,120 rows, 5 columns
    ['order_id', 'line_no', 'item_name', 'quantity', 'unit_price']
payments: 16,423 rows, 5 columns
    ['payment_id', 'order_id', 'payment_type', 'amount', 'paid_at']
restaurants: 67 rows, 5 columns
    ['restaurant_id', 'restaurant_name', 'cuisine', 'city', 'effective_from']
customers: 4,200 rows, 3 columns
    ['customer_id', 'signup_date', 'home_city']


## Examples: two ways to work with the tables

You can use SQL, pandas, or a mix of both. The next two cells show each, using only the tables and keys described above. Run them and change them to see what they do. You do not have to use them in your answer.

In [ ]:
# Example 1: SQL
# q() runs SQL against any of the tables above and returns a pandas DataFrame.
from IPython.display import display

# Look at the first few rows of a table
display(q("SELECT * FROM orders LIMIT 5"))

# How many rows are there, and how many different order_id values?
display(q("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT order_id) AS n_distinct_orders FROM orders"))

# Join two tables on a key
display(q("""
    SELECT o.order_id, r.restaurant_id
    FROM orders AS o
    JOIN restaurants AS r ON o.restaurant_id = r.restaurant_id
    LIMIT 5
"""))

,order_id,customer_id,restaurant_id,order_ts,delivery_fee
0,O000001,C01161,R052,2026-03-01T09:13:23Z,1.49
1,O000002,C00723,R027,2026-03-01T09:15:33Z,2.49
2,O000003,C00781,R044,2026-03-01T09:23:42Z,0.00
3,O000004,C01430,R049,2026-03-01T09:28:14Z,1.49
4,O000005,C00039,R009,2026-03-01T11:32:35Z,0.00


,n_rows,n_distinct_orders
0,14494,14494


,order_id,restaurant_id
0,O000001,R052
1,O000002,R027
2,O000003,R044
3,O000004,R049
4,O000005,R009


In [ ]:
# Example 2: pandas
# Each table is also a pandas DataFrame, stored in the dictionary `tables`.
from IPython.display import display

orders = tables["orders"]
display(orders.head())        # first few rows
print(orders.shape)           # (number of rows, number of columns)
print(orders.dtypes)          # data type of each column

# The same join as in Example 1, using pandas
joined = tables["orders"].merge(tables["restaurants"], on="restaurant_id", how="inner")
display(joined[["order_id", "restaurant_id"]].head())

# You can mix the two: run SQL, then use pandas methods on the result
result = q("SELECT customer_id FROM orders")
print(result["customer_id"].nunique(), "different customer_id values in orders")

,order_id,customer_id,restaurant_id,order_ts,delivery_fee
0,O000001,C01161,R052,2026-03-01T09:13:23Z,1.49
1,O000002,C00723,R027,2026-03-01T09:15:33Z,2.49
2,O000003,C00781,R044,2026-03-01T09:23:42Z,0.00
3,O000004,C01430,R049,2026-03-01T09:28:14Z,1.49
4,O000005,C00039,R009,2026-03-01T11:32:35Z,0.00


(14494, 5)
order_id          object
customer_id       object
restaurant_id     object
order_ts          object
delivery_fee     float64
dtype: object


,order_id,restaurant_id
0,O000001,R052
1,O000002,R027
2,O000003,R044
3,O000004,R049
4,O000005,R009


4073 different customer_id values in orders


## Your brief

**From:** Operations

> We run Plateful, a food-delivery marketplace in five UK cities. We are starting a weekly operations report and need the first layers of a data pipeline behind it: clean tables first, then one reporting table. For each week, show orders, net revenue and average delivery time by cuisine and by city. We also need to know how far we can trust the figures.

**What your final table should look like**

One table with columns for week, cuisine, city, number of orders, net revenue and average delivery time in minutes. State clearly what one row of your table represents.

**Decisions you must make and state**

- Which week does an order belong to?
- Which orders count as orders, and which do not?
- What does "net revenue" mean here, and where does it come from?
- What counts as "delivery time", and for which orders can it be measured?

**A suggested route, in order** (you are not expected to finish)

1. Profile the tables: grain, keys, duplicates, odd values.
2. Build clean tables (a silver layer) with one row per order where that is the grain.
3. Build the one reporting table (gold) from them.
4. Show with checks that the gold table agrees with the source data.
5. If you have time: add cancelled orders and a refund rate to the table, and write automated checks for your clean tables.

## 1. Understand the data

*Look before you build. Write down what you find: what one row of each table represents, how the tables connect, and anything that looks odd or missing.*

Before I start cleaning the data, I want to see the sample of the raw data first

PROMPT : show first 5 rows of all 5 tables

In [ ]:
for table_name in tables.keys():
    print(f"\n--- First 5 rows of: {table_name} ---")
    display(q(f"SELECT * FROM {table_name} LIMIT 5"))


--- First 5 rows of: orders ---


,order_id,customer_id,restaurant_id,order_ts,delivery_fee
0,O000001,C01161,R052,2026-03-01T09:13:23Z,1.49
1,O000002,C00723,R027,2026-03-01T09:15:33Z,2.49
2,O000003,C00781,R044,2026-03-01T09:23:42Z,0.00
3,O000004,C01430,R049,2026-03-01T09:28:14Z,1.49
4,O000005,C00039,R009,2026-03-01T11:32:35Z,0.00



--- First 5 rows of: order_status_events ---


,event_id,order_id,status,event_ts
0,E027809,O007104,accepted,2026-04-29T13:28:57Z
1,E018209,O004658,placed,2026-04-08T19:11:30Z
2,E043792,O011178,accepted,2026-06-02T17:07:47Z
3,E021428,O005468,delivered,2026-04-15T13:43:36Z
4,E037134,O009483,picked_up,2026-05-19T12:18:18Z



--- First 5 rows of: order_lines ---


,order_id,line_no,item_name,quantity,unit_price
0,O000001,1,Ramen,3,12.0
1,O000002,1,Soft drink,1,2.0
2,O000002,2,Side salad,1,3.5
3,O000002,3,Burrito,3,8.0
4,O000003,1,Fries,1,3.0


... (2 outputs left out)


--- First 5 rows of: restaurants ---


,restaurant_id,restaurant_name,cuisine,city,effective_from
0,R064,Fresh Vegan 64,Vegan,London,2025-01-01
1,R012,Royal Burgers 12,Burgers,Bristol,2025-01-01
2,R028,Grand Burgers 28,Burgers,Birmingham,2025-01-01
3,R007,Royal Japanese 7,Japanese,Bristol,2025-01-01
4,R026,Corner Indian 26,Indian,London,2025-01-01



--- First 5 rows of: customers ---


,customer_id,signup_date,home_city
0,C00001,2024-10-15,London
1,C00002,2025-05-17,Leeds
2,C00003,2026-01-25,Leeds
3,C00004,2024-09-14,London
4,C00005,2024-01-25,Manchester


Since there are quite a lot of data to go through and check for each table, I plan to check and clean by table before moving on to the next.

Let start with orders table. I want to first check for any missing value

PROMPT : check for any missing value in table orders

In [ ]:
# Check for missing (null) values in the orders table using DuckDB
q("""
SELECT
    COUNT(*) as total_rows,
    COUNT(order_id) as non_null_order_id,
    COUNT(customer_id) as non_null_customer_id,
    COUNT(restaurant_id) as non_null_restaurant_id,
    COUNT(order_ts) as non_null_order_ts,
    COUNT(delivery_fee) as non_null_delivery_fee
FROM orders
""")

,total_rows,non_null_order_id,non_null_customer_id,non_null_restaurant_id,non_null_order_ts,non_null_delivery_fee
0,14494,14494,14494,14494,14494,14494


Based on the result I got there are no NA value in the orders table since the number of non null value for all the columns are the same as total row

So I will start checking for any duplication in this table especially for the id columns

PROMPT : check for data duplication in orders table and also check for duplication in order_id, customer_id, restaurant_id

In [ ]:
# Check for exact duplicate rows in the entire orders table using a subquery
print("Total fully duplicated rows in orders:")
display(q("""
SELECT (SELECT COUNT(*) FROM orders) - (SELECT COUNT(*) FROM (SELECT DISTINCT * FROM orders)) as duplicate_rows
"""))

# Check the unique count and duplicates for order_id, customer_id, and restaurant_id
print("\nUniqueness analysis:")
display(q("""
SELECT
    COUNT(order_id) as total_rows,
    COUNT(DISTINCT order_id) as unique_order_ids,
    COUNT(DISTINCT customer_id) as unique_customer_ids,
    COUNT(DISTINCT restaurant_id) as unique_restaurant_ids
FROM orders
"""))

Total fully duplicated rows in orders:


,duplicate_rows
0,0



Uniqueness analysis:


,total_rows,unique_order_ids,unique_customer_ids,unique_restaurant_ids
0,14494,14494,4073,64


Okay so there is no exact duplication in this table and that all the order_id are unique which is expected for the orders table

I think next I will check for the formating for all the column whether it's consistent

PROMPT : check the formatting and consistency for all the column in table orders

In [ ]:
# Check min and max values for timestamps, delivery fees, and length profiles of IDs
q("""
SELECT
    MIN(order_ts) as min_order_ts,
    MAX(order_ts) as max_order_ts,
    MIN(delivery_fee) as min_delivery_fee,
    MAX(delivery_fee) as max_delivery_fee,
    COUNT(CASE WHEN LENGTH(order_id) != 7 THEN 1 END) as invalid_order_id_len_count,
    COUNT(CASE WHEN LENGTH(customer_id) != 6 THEN 1 END) as invalid_customer_id_len_count,
    COUNT(CASE WHEN LENGTH(restaurant_id) != 4 THEN 1 END) as invalid_restaurant_id_len_count
FROM orders
""")

,min_order_ts,max_order_ts,min_delivery_fee,max_delivery_fee,invalid_order_id_len_count,invalid_customer_id_len_count,invalid_restaurant_id_len_count
0,2026-03-01T09:13:23Z,2026-06-30T22:58:46Z,0.0,3.49,0,0,0


PROMPT (to clarify) : I mean the format of how each ID being written or how the time being written or like for delivery fee check for the statistic

In [ ]:
# 1. Verify specific prefix formats for our IDs using DuckDB regex operator (~)
print("--- ID Pattern and Prefix Verification ---")
display(q("""
SELECT
    COUNT(CASE WHEN order_id ~ '^O\\d{6}$' THEN 1 END) as valid_order_ids,
    COUNT(CASE WHEN customer_id ~ '^C\\d{5}$' THEN 1 END) as valid_customer_ids,
    COUNT(CASE WHEN restaurant_id ~ '^R\\d{3}$' THEN 1 END) as valid_restaurant_ids,
    COUNT(*) as total_rows
FROM orders
"""))

# 2. Check if order_ts values strictly follow the timestamp pattern
print("\n--- Timestamp Formatting Check ---")
display(q("""
SELECT
    COUNT(CASE WHEN order_ts ~ '^\\d{4}-\\d{2}-\\d{2}T\\d{2}:\\d{2}:\\d{2}Z$' THEN 1 END) as valid_utc_timestamps,
    COUNT(*) as total_rows
FROM orders
"""))

# 3. Descriptive statistics for delivery_fee
print("\n--- Delivery Fee Descriptive Statistics ---")
display(tables['orders']['delivery_fee'].describe().to_frame())

--- ID Pattern and Prefix Verification ---


,valid_order_ids,valid_customer_ids,valid_restaurant_ids,total_rows
0,14494,14494,14494,14494



--- Timestamp Formatting Check ---


,valid_utc_timestamps,total_rows
0,14494,14494



--- Delivery Fee Descriptive Statistics ---


,delivery_fee
count,14494.000000
mean,2.150676
std,1.104824
min,0.000000
25%,1.490000
50%,2.490000
75%,2.990000
max,3.490000


Okay everything looks normal. I will move on to the next table which is order_status_event

Will also start with checking for missing value

PROMPT : check for NA for table order_status_event

In [ ]:
print("--- Missing Value Check for order_status_events ---")
display(q("""
SELECT
    COUNT(*) as total_rows,
    COUNT(event_id) as non_null_event_id,
    COUNT(order_id) as non_null_order_id,
    COUNT(status) as non_null_status,
    COUNT(event_ts) as non_null_event_ts
FROM order_status_events
"""))

--- Missing Value Check for order_status_events ---


,total_rows,non_null_event_id,non_null_order_id,non_null_status,non_null_event_ts
0,56812,56812,56812,56812,56812


Okay all the column have the same number of non null value as the total row so there are no missing value in this table

Next will check for data duplication and also duplication in id

PROMPT : check for data duplication in table order_status_events and also check for duplication in event_id and order_id in this table

In [ ]:
# 1. Check for exact duplicate rows in the entire order_status_events table
print("Total fully duplicated rows in order_status_events:")
display(q("""
SELECT (SELECT COUNT(*) FROM order_status_events) - (SELECT COUNT(*) FROM (SELECT DISTINCT * FROM order_status_events)) as duplicate_rows
"""))

# 2. Check uniqueness of event_id and count how many orders are present
print("\nUniqueness analysis for event_id and order_id:")
display(q("""
SELECT
    COUNT(event_id) as total_events,
    COUNT(DISTINCT event_id) as unique_event_ids,
    COUNT(DISTINCT order_id) as unique_order_ids,
    COUNT(event_id) - COUNT(DISTINCT event_id) as duplicate_event_ids
FROM order_status_events
"""))

# 3. Look at distribution of how many events exist per order
print("\nDistribution of event counts per order:")
display(q("""
SELECT
    events_per_order,
    COUNT(*) as count_of_orders
FROM (
    SELECT order_id, COUNT(*) as events_per_order
    FROM order_status_events
    GROUP BY order_id
)
GROUP BY events_per_order
ORDER BY events_per_order ASC
"""))

Total fully duplicated rows in order_status_events:


,duplicate_rows
0,0



Uniqueness analysis for event_id and order_id:


,total_events,unique_event_ids,unique_order_ids,duplicate_event_ids
0,56812,56812,14494,0



Distribution of event counts per order:


,events_per_order,count_of_orders
0,2,439
1,3,643
2,4,13058
3,5,351
4,6,3


There are no exact duplication row in this table and also the event_id are all unique for each of the event which is expected for this table while there can be a duplication for order id which is expected

Next let check for the formating

PROMPT : check for the formating for all the column for table order_status_events

In [ ]:
# 1. Verify specific ID format patterns
print("--- ID Format Pattern Checks ---")
display(q("""
SELECT
    COUNT(CASE WHEN event_id ~ '^E\\d{6}$' THEN 1 END) as valid_event_ids,
    COUNT(CASE WHEN order_id ~ '^O\\d{6}$' THEN 1 END) as valid_order_ids,
    COUNT(*) as total_rows
FROM order_status_events
"""))

# 2. Check the set of distinct status values
print("\n--- Unique Status Values ---")
display(q("""
SELECT DISTINCT status
FROM order_status_events
"""))

# 3. Check if event_ts strictly matches ISO-8601 UTC timestamp format
print("\n--- Timestamp Format Checks ---")
display(q("""
SELECT
    COUNT(CASE WHEN event_ts ~ '^\\d{4}-\\d{2}-\\d{2}T\\d{2}:\\d{2}:\\d{2}Z$' THEN 1 END) as valid_utc_timestamps,
    COUNT(*) as total_rows
FROM order_status_events
"""))

--- ID Format Pattern Checks ---


,valid_event_ids,valid_order_ids,total_rows
0,56812,56812,56812



--- Unique Status Values ---


,status
0,placed
1,delivered
2,picked_up
3,accepted
4,cancelled



--- Timestamp Format Checks ---


,valid_utc_timestamps,total_rows
0,56812,56812


All the formating checkout so will move on to the next table which is order_lines

Will start of by checking for missing value

RPOMPT ; check for missing value in table order_lines

In [ ]:
print("--- Missing Value Check for order_lines ---")
display(q("""
SELECT
    COUNT(*) as total_rows,
    COUNT(order_id) as non_null_order_id,
    COUNT(line_no) as non_null_line_no,
    COUNT(item_name) as non_null_item_name,
    COUNT(quantity) as non_null_quantity,
    COUNT(unit_price) as non_null_unit_price
FROM order_lines
"""))

--- Missing Value Check for order_lines ---


,total_rows,non_null_order_id,non_null_line_no,non_null_item_name,non_null_quantity,non_null_unit_price
0,36120,36120,36120,36120,36120,36120


Okay there are also no missing value in this table which is expected

will quickly moveon to check for the data duplication

PROMPT : check for data duplication in table order_lines and also duplication in column order_id and line_no for this table

In [ ]:
# 1. Check for exact duplicate rows in the entire order_lines table
print("Total fully duplicated rows in order_lines:")
display(q("""
SELECT (SELECT COUNT(*) FROM order_lines) - (SELECT COUNT(*) FROM (SELECT DISTINCT * FROM order_lines)) as duplicate_rows
"""))

# 2. Check if the natural key (order_id, line_no) is perfectly unique
print("\nChecking uniqueness of the compound key (order_id, line_no):")
display(q("""
SELECT
    COUNT(*) as total_rows,
    COUNT(DISTINCT order_id || '-' || CAST(line_no AS VARCHAR)) as unique_order_line_combinations
FROM order_lines
"""))

# 3. Analyze the distribution of how many order lines (items) exist per order_id
print("\nDistribution of line items per order:")
display(q("""
SELECT
    lines_per_order,
    COUNT(*) as count_of_orders
FROM (
    SELECT order_id, COUNT(*) as lines_per_order
    FROM order_lines
    GROUP BY order_id
)
GROUP BY lines_per_order
ORDER BY lines_per_order ASC
"""))

Total fully duplicated rows in order_lines:


,duplicate_rows
0,0



Checking uniqueness of the compound key (order_id, line_no):


,total_rows,unique_order_line_combinations
0,36120,36120



Distribution of line items per order:


,lines_per_order,count_of_orders
0,1,3635
1,2,3680
2,3,3591
3,4,3588


okay there are no duplication rows and also no duplication in the order_id and line_no combination

next will check for the formating

PROMPT : check for the formating for each column in tbale order_lines

In [ ]:
# 1. Check ID formats and verify that line_no and quantity are positive integers
print("--- Pattern and Integrity Verification ---")
display(q("""
SELECT
    COUNT(CASE WHEN order_id ~ '^O\\d{6}$' THEN 1 END) as valid_order_ids,
    COUNT(CASE WHEN line_no > 0 THEN 1 END) as positive_line_numbers,
    COUNT(CASE WHEN quantity > 0 THEN 1 END) as positive_quantities,
    COUNT(*) as total_rows
FROM order_lines
"""))

# 2. Summary stats for quantity and unit_price
print("\n--- Numeric Descriptive Statistics ---")
display(tables['order_lines'][['quantity', 'unit_price']].describe())

# 3. Sample of some unique item names to make sure they are valid strings
print("\n--- Sample of Unique Item Names (First 10) ---")
display(q("""
SELECT item_name, COUNT(*) as occurrences
FROM order_lines
GROUP BY item_name
LIMIT 10
"""))

--- Pattern and Integrity Verification ---


,valid_order_ids,positive_line_numbers,positive_quantities,total_rows
0,36120,36120,36120,36120



--- Numeric Descriptive Statistics ---


,quantity,unit_price
count,36120.000000,36120.000000
mean,1.995903,7.211739
std,0.813985,3.131357
min,1.000000,2.000000
25%,1.000000,5.000000
50%,2.000000,8.000000
75%,3.000000,10.500000
max,3.000000,12.000000



--- Sample of Unique Item Names (First 10) ---


,item_name,occurrences
0,Side salad,2976
1,Burrito,3008
2,Ramen,2967
3,Soft drink,3034
4,Fries,2982
5,Fried rice,3022
6,Chicken tikka,3005
7,Pad thai,3101
8,Dessert,3054
9,Margherita,2994


Nothing seems out of ordinary for me

Will move on to the next table which is payment

PROMPT: check for missing value in table payment

In [ ]:
print("--- Missing Value Check for payments ---")
display(q("""
SELECT
    COUNT(*) as total_rows,
    COUNT(payment_id) as non_null_payment_id,
    COUNT(order_id) as non_null_order_id,
    COUNT(payment_type) as non_null_payment_type,
    COUNT(amount) as non_null_amount,
    COUNT(paid_at) as non_null_paid_at
FROM payments
"""))

--- Missing Value Check for payments ---


,total_rows,non_null_payment_id,non_null_order_id,non_null_payment_type,non_null_amount,non_null_paid_at
0,16423,16423,16423,16423,16423,16423


there are no missing value in table payment so will move on to check the dupliation in data and in payment id

PROMPT : check for data duplication in the table payement and in column payment_id

In [ ]:
# 1. Check for exact duplicate rows in the entire payments table
print("Total fully duplicated rows in payments:")
display(q("""
SELECT (SELECT COUNT(*) FROM payments) - (SELECT COUNT(*) FROM (SELECT DISTINCT * FROM payments)) as duplicate_rows
"""))

# 2. Check uniqueness of payment_id and examine order_id duplication
print("\nUniqueness analysis for payment_id and order_id:")
display(q("""
SELECT
    COUNT(payment_id) as total_payments,
    COUNT(DISTINCT payment_id) as unique_payment_ids,
    COUNT(DISTINCT order_id) as unique_order_ids,
    COUNT(payment_id) - COUNT(DISTINCT payment_id) as duplicate_payment_ids
FROM payments
"""))

Total fully duplicated rows in payments:


,duplicate_rows
0,30



Uniqueness analysis for payment_id and order_id:


,total_payments,unique_payment_ids,unique_order_ids,duplicate_payment_ids
0,16423,16393,14494,30


there are 30 duplicates row and duplicates id

I will pull it out to investigate more

PROMPT : pull out the 30 duplicates row in table payment

In [ ]:
# Retrieve and display the duplicate rows in the payments table
duplicates_df = q("""
WITH duplicated_ids AS (
    SELECT payment_id
    FROM payments
    GROUP BY payment_id
    HAVING COUNT(*) > 1
)
SELECT *
FROM payments
WHERE payment_id IN (SELECT payment_id FROM duplicated_ids)
ORDER BY payment_id
""")

display(duplicates_df)

,payment_id,order_id,payment_type,amount,paid_at
0,P000219,O000185,charge,25.49,2026-03-02 19:05:43
1,P000219,O000185,charge,25.49,2026-03-02 19:05:43
2,P000927,O000815,charge,23.99,2026-03-08 11:53:53
3,P000927,O000815,charge,23.99,2026-03-08 11:53:53
4,P000998,O000874,charge,50.99,2026-03-08 19:19:26
5,P000998,O000874,charge,50.99,2026-03-08 19:19:26
6,P001859,O001631,charge,38.49,2026-03-14 22:43:50
7,P001859,O001631,charge,38.49,2026-03-14 22:43:50
8,P002189,O001926,charge,41.99,2026-03-17 18:22:24
9,P002189,O001926,charge,41.99,2026-03-17 18:22:24


From quick check it seems to be exact duplication so I will remove all the duplicate date

RPOMPT : remove all the exact duplicate row in table payment and run the check that there are no duplicate row left

In [ ]:
# Remove exact duplicates by pulling only unique records
deduplicated_payments_df = q("SELECT DISTINCT * FROM payments")

# Re-register the deduplicated dataframe as the 'payments' table in DuckDB
con.register('payments', deduplicated_payments_df)
tables['payments'] = deduplicated_payments_df

# Verify that no duplicate payment_ids remain
verification_df = q("""
SELECT
    COUNT(*) as total_rows,
    COUNT(DISTINCT payment_id) as unique_payment_ids,
    COUNT(*) - COUNT(DISTINCT payment_id) as remaining_duplicates
FROM payments
""")

display(verification_df)

,total_rows,unique_payment_ids,remaining_duplicates
0,16393,16393,0


okay I have remove the duplicate row and verify that it have been remove

Next will check for the formating

PROMPT : check the formating of the data in table payment

In [ ]:
# 1. Verify specific ID format patterns for payments
print("--- ID Format Pattern Checks for Payments ---")
display(q("""
SELECT
    COUNT(CASE WHEN payment_id ~ '^P\\d{6}$' THEN 1 END) as valid_payment_ids,
    COUNT(CASE WHEN order_id ~ '^O\\d{6}$' THEN 1 END) as valid_order_ids,
    COUNT(*) as total_rows
FROM payments
"""))

# 2. Check the set of distinct payment_type values
print("\\n--- Unique Payment Type Values ---")
display(q("""
SELECT payment_type, COUNT(*) as occurrences
FROM payments
GROUP BY payment_type
"""))

# 3. Check if paid_at strictly matches standard timestamp format
print("\\n--- Timestamp Format Checks for Payments ---")
display(q("""
SELECT
    COUNT(CASE WHEN paid_at ~ '^\\d{4}-\\d{2}-\\d{2} \\d{2}:\\d{2}:\\d{2}$' THEN 1 END) as valid_timestamps,
    COUNT(*) as total_rows
FROM payments
"""))

# 4. Summary stats for payment amount
print("\\n--- Payment Amount Descriptive Statistics ---")
display(tables['payments']['amount'].describe().to_frame())

--- ID Format Pattern Checks for Payments ---


,valid_payment_ids,valid_order_ids,total_rows
0,16393,16393,16393


\n--- Unique Payment Type Values ---


,payment_type,occurrences
0,charge,14494
1,refund,1899


\n--- Timestamp Format Checks for Payments ---


,valid_timestamps,total_rows
0,16393,16393


\n--- Payment Amount Descriptive Statistics ---


,amount
count,16393.000000
mean,30.975435
std,29.094554
min,-127.490000
25%,14.490000
50%,31.490000
75%,49.990000
max,127.490000


payment amount seems usual for me since the minimun value is negative so will pull out the amount less than 0 to check

PROMPT : filter out the amount value less than 0 in table payment

In [ ]:
# Retrieve and display payment records with negative amounts
negative_payments_df = q("""
SELECT *
FROM payments
WHERE amount < 0
ORDER BY amount ASC
""")

display(negative_payments_df)
print(f"Total negative payment rows found: {len(negative_payments_df)}")

,payment_id,order_id,payment_type,amount,paid_at
0,P013254,O011718,refund,-127.49,2026-06-07 11:39:39
1,P016298,O014407,refund,-105.99,2026-06-30 11:27:03
2,P002327,O002048,refund,-102.99,2026-03-18 19:23:07
3,P005682,O005025,refund,-101.99,2026-04-11 18:34:00
4,P002191,O001927,refund,-101.99,2026-03-17 18:41:53
...,...,...,...,...,...
1894,P006404,O005668,refund,-0.44,2026-04-17 15:20:20
1895,P002534,O002242,refund,-0.42,2026-03-21 00:29:11
1896,P011321,O010007,refund,-0.40,2026-05-25 11:48:28
1897,P000124,O000106,refund,-0.38,2026-03-03 16:33:05


Total negative payment rows found: 1899


oh so it's seems to be refund which make sense. Will quickly run code again that all the negative amount has payment_type refund

PROMPT : filter all the payment_type for amount less than 0 in table payment

In [ ]:
# Verify the distinct payment_types for rows where amount is less than 0
refund_types_df = q("""
SELECT DISTINCT payment_type, COUNT(*) as count
FROM payments
WHERE amount < 0
GROUP BY payment_type
""")

display(refund_types_df)

,payment_type,count
0,refund,1899


Okay so it confirm so will move on to the next table which is restaurants

will start with missing value

PROMPT : check for any missing value in table restaurants

In [ ]:
# Check for missing (null) values in the restaurants table
print("--- Missing Value Check for restaurants ---")
display(q("""
SELECT
    COUNT(*) as total_rows,
    COUNT(restaurant_id) as non_null_restaurant_id,
    COUNT(restaurant_name) as non_null_restaurant_name,
    COUNT(cuisine) as non_null_cuisine,
    COUNT(city) as non_null_city,
    COUNT(effective_from) as non_null_effective_from
FROM restaurants
"""))

--- Missing Value Check for restaurants ---


,total_rows,non_null_restaurant_id,non_null_restaurant_name,non_null_cuisine,non_null_city,non_null_effective_from
0,67,67,67,67,67,67


there is no missing value so will start checking for duplication

PROMPT : check for data duplication in table restaurants and also duplication in the restaurant_id

In [ ]:
# 1. Check for exact duplicate rows in the entire restaurants table
print("Total fully duplicated rows in restaurants:")
display(q("""
SELECT (SELECT COUNT(*) FROM restaurants) - (SELECT COUNT(*) FROM (SELECT DISTINCT * FROM restaurants)) as duplicate_rows
"""))

# 2. Check the uniqueness of restaurant_id
print("\nUniqueness analysis for restaurant_id:")
display(q("""
SELECT
    COUNT(restaurant_id) as total_rows,
    COUNT(DISTINCT restaurant_id) as unique_restaurant_ids,
    COUNT(restaurant_id) - COUNT(DISTINCT restaurant_id) as duplicate_restaurant_ids
FROM restaurants
"""))

Total fully duplicated rows in restaurants:


,duplicate_rows
0,0



Uniqueness analysis for restaurant_id:


,total_rows,unique_restaurant_ids,duplicate_restaurant_ids
0,67,64,3


there is no exact duplicate row but there are duplicate restaurant_id so will filter out to investigate more

PROMPT : filter out the row in table reataurant with duplicate restaurant_id

In [ ]:
# Retrieve and display restaurant records with duplicate restaurant_id
duplicate_restaurants = q("""
WITH duplicated_ids AS (
    SELECT restaurant_id
    FROM restaurants
    GROUP BY restaurant_id
    HAVING COUNT(*) > 1
)
SELECT *
FROM restaurants
WHERE restaurant_id IN (SELECT restaurant_id FROM duplicated_ids)
ORDER BY restaurant_id, effective_from
""")

display(duplicate_restaurants)
print(f"Total rows belonging to duplicate restaurant IDs: {len(duplicate_restaurants)}")

,restaurant_id,restaurant_name,cuisine,city,effective_from
0,R007,Royal Japanese 7,Japanese,Bristol,2025-01-01
1,R007,Royal Japanese 7,Japanese,London,2026-04-14
2,R021,Grand Thai 21,Thai,Manchester,2025-01-01
3,R021,Grand Thai 21,Thai,LEEDS,2026-05-06
4,R040,Corner Vegan 40,Vegan,London,2025-01-01
5,R040,Corner Vegan 40,Vegan,Bristol,2026-05-27


Total rows belonging to duplicate restaurant IDs: 6


From the result, it seems likee these 3 restaurant change their location so that why one restaurant_id can have more than one location

So I will only keep the most recent record

PROMPT : remove duplication in the restaurant_id in table restaurant by only keeping the most recent record as well as verify that there is no duplication remain after the cleaning

In [ ]:
cleaned_restaurants_df = q("""
WITH ranked_restaurants AS (
    SELECT
        restaurant_id,
        restaurant_name,
        cuisine,
        -- Correctly capitalize the first letter and lowercase the rest in DuckDB
        UPPER(SUBSTR(city, 1, 1)) || LOWER(SUBSTR(city, 2)) AS city,
        effective_from,
        ROW_NUMBER() OVER (
            PARTITION BY restaurant_id
            ORDER BY effective_from DESC
        ) as rn
    FROM restaurants
)
SELECT
    restaurant_id,
    restaurant_name,
    cuisine,
    city,
    effective_from
FROM ranked_restaurants
WHERE rn = 1
""")

# Register the cleaned table back into DuckDB and tables dictionary
con.register('restaurants', cleaned_restaurants_df)
tables['restaurants'] = cleaned_restaurants_df

# Verify that no duplicate restaurant_ids remain
verification_restaurants = q("""
SELECT
    COUNT(*) as total_rows,
    COUNT(DISTINCT restaurant_id) as unique_restaurant_ids,
    COUNT(*) - COUNT(DISTINCT restaurant_id) as remaining_duplicates
FROM restaurants
""")

display(verification_restaurants)
display(q("SELECT * FROM restaurants WHERE restaurant_id IN ('R007', 'R021', 'R040')"))

,total_rows,unique_restaurant_ids,remaining_duplicates
0,64,64,0


,restaurant_id,restaurant_name,cuisine,city,effective_from
0,R007,Royal Japanese 7,Japanese,London,2026-04-14
1,R021,Grand Thai 21,Thai,Leeds,2026-05-06
2,R040,Corner Vegan 40,Vegan,Bristol,2026-05-27


Okay I remove the duplication and verify that there are no duplication left so will move on to check the formating

PROMPT : check the formating of each column in table restaurant

In [ ]:
# 1. Verify restaurant_id formatting pattern (R followed by 3 digits)
print("--- ID Format Pattern Checks ---")
display(q("""
SELECT
    COUNT(CASE WHEN restaurant_id ~ '^R\\d{3}$' THEN 1 END) as valid_restaurant_ids,
    COUNT(*) as total_rows
FROM restaurants
"""))

# 2. Check the set of unique cuisines and their frequency
print("\n--- Unique Cuisine Values and Counts ---")
display(q("""
SELECT cuisine, COUNT(*) as occurrences
FROM restaurants
GROUP BY cuisine
ORDER BY occurrences DESC
"""))

# 3. Check the set of unique cities
print("\n--- Unique City Values and Counts ---")
display(q("""
SELECT city, COUNT(*) as occurrences
FROM restaurants
GROUP BY city
ORDER BY occurrences DESC
"""))

# 4. Check if effective_from strictly matches UTC date or timestamp patterns
print("\n--- Date/Timestamp Format Checks ---")
display(q("""
SELECT
    COUNT(CASE WHEN effective_from ~ '^\\d{4}-\\d{2}-\\d{2}$' THEN 1 END) as valid_dates,
    COUNT(*) as total_rows
FROM restaurants
"""))

--- ID Format Pattern Checks ---


,valid_restaurant_ids,total_rows
0,64,64



--- Unique Cuisine Values and Counts ---


,cuisine,occurrences
0,Chinese,8
1,Vegan,8
2,Italian,7
3,Burgers,7
4,Thai,7
5,Indian,7
6,Japanese,6
7,Mexican,6
8,mexican,2
9,Burgers,1



--- Unique City Values and Counts ---


,city,occurrences
0,London,23
1,Manchester,17
2,Birmingham,12
3,Leeds,6
4,Bristol,6



--- Date/Timestamp Format Checks ---


,valid_dates,total_rows
0,64,64


There are mix of formating in cuisine of the case and white space issue so will clean that up first

PROMPT : fix the formating in column cuisine in table restaurant to so the all the name start with capital letter and remove all the whitespace before and after the word

In [ ]:
cleaned_restaurants_df = q("""
SELECT
    restaurant_id,
    restaurant_name,
    -- Trim whitespaces and apply Title Case-like formatting to the cuisine column
    UPPER(SUBSTR(TRIM(cuisine), 1, 1)) || LOWER(SUBSTR(TRIM(cuisine), 2)) AS cuisine,
    city,
    effective_from
FROM restaurants
""")

# Register the cleaned table back into DuckDB and tables dictionary
con.register('restaurants', cleaned_restaurants_df)
tables['restaurants'] = cleaned_restaurants_df

# Verify that the cuisine column is cleaned
print("--- Cleaned Cuisine Values and Counts ---")
display(q("""
SELECT cuisine, COUNT(*) as occurrences
FROM restaurants
GROUP BY cuisine
ORDER BY occurrences DESC
"""))

--- Cleaned Cuisine Values and Counts ---


,cuisine,occurrences
0,Italian,8
1,Japanese,8
2,Burgers,8
3,Mexican,8
4,Chinese,8
5,Thai,8
6,Vegan,8
7,Indian,8


okay the new cuisine type look okay now so will quickly move on to the last table due to time limit

PROMPT : check for missing value in table customer

In [ ]:
# Check for missing (null) values in the customers table using DuckDB
print("--- Missing Value Check for customers ---")
display(q("""
SELECT
    COUNT(*) as total_rows,
    COUNT(customer_id) as non_null_customer_id,
    COUNT(signup_date) as non_null_signup_date,
    COUNT(home_city) as non_null_home_city
FROM customers
"""))

--- Missing Value Check for customers ---


,total_rows,non_null_customer_id,non_null_signup_date,non_null_home_city
0,4200,4200,4200,4200


no missing value so will start checking for duplication

PROMPT : check for data duplication in table customers and check for duplication in the column customer_id

In [ ]:
# 1. Check for exact duplicate rows in the entire customers table
print("Total fully duplicated rows in customers:")
display(q("""
SELECT (SELECT COUNT(*) FROM customers) - (SELECT COUNT(*) FROM (SELECT DISTINCT * FROM customers)) as duplicate_rows
"""))

# 2. Check the uniqueness of customer_id
print("\nUniqueness analysis for customer_id:")
display(q("""
SELECT
    COUNT(customer_id) as total_rows,
    COUNT(DISTINCT customer_id) as unique_customer_ids,
    COUNT(customer_id) - COUNT(DISTINCT customer_id) as duplicate_customer_ids
FROM customers
"""))

Total fully duplicated rows in customers:


,duplicate_rows
0,0



Uniqueness analysis for customer_id:


,total_rows,unique_customer_ids,duplicate_customer_ids
0,4200,4200,0


no duplication in the row and in customer_id so will check the formating now

PROMPT : check for the formating of the value in each column in table customers

In [ ]:
# 1. Verify pattern correctness for customer_id and signup_date
print("--- Pattern Verification for Customers ---")
display(q("""
SELECT
    COUNT(CASE WHEN customer_id ~ '^C\\d{5}$' THEN 1 END) as valid_customer_ids,
    COUNT(CASE WHEN signup_date ~ '^\\d{4}-\\d{2}-\\d{2}$' THEN 1 END) as valid_signup_dates,
    COUNT(*) as total_rows
FROM customers
"""))

# 2. Check the unique cities in the table to spot any duplicates with different formatting/casing
print("\n--- Unique Home Cities and Counts ---")
display(q("""
SELECT home_city, COUNT(*) as occurrences
FROM customers
GROUP BY home_city
ORDER BY occurrences DESC
"""))

--- Pattern Verification for Customers ---


,valid_customer_ids,valid_signup_dates,total_rows
0,4200,4200,4200



--- Unique Home Cities and Counts ---


,home_city,occurrences
0,London,1541
1,Manchester,857
2,Birmingham,697
3,Bristol,569
4,Leeds,536


Okay everything looks good

## 2. Plan

*Before you write the main code, write your plan in your own words: what one row of your final table is, which tables you will join and in what order, how many rows you expect at each step, and how you will handle the problems you found.*

Brief
One table with columns for week, cuisine, city, number of orders, net revenue and average delivery time in minutes. State clearly what one row of your table represents.

will move on to the build first due to the time limit

## 3. Build

*Build as far as you can in the time. Short, re-runnable cells are better than one long cell.*

Add text here as you go along. Show that you are staying in control of the build!

due to time limit will ask AI to help create table for this brief using data that we clean already

PROMPT : help create one table with columns for week, cuisine, city, number of orders, net revenue and average delivery time in minutes using the data that we already clean

### Weekly Operations Pipeline Execution

We will define our metrics as follows:
1. **Week**: The Monday of the week in which the order was placed (`DATE_TRUNC('week', CAST(order_ts AS TIMESTAMP))`).
2. **Net Revenue**: The sum of all payments (charges + refunds) associated with the order (`amount`).
3. **Delivery Time**: The difference in minutes between the `'placed'` event and the `'delivered'` event in `order_status_events`.
4. **Grain**: One row in the final table represents a combination of **Week**, **Cuisine**, and **City**.

In [ ]:
# Calculate metrics per order first to construct the silver layer
order_metrics_df = q("""
WITH order_times AS (
    SELECT
        order_id,
        MIN(CASE WHEN status = 'placed' THEN CAST(event_ts AS TIMESTAMP) END) as placed_at,
        MIN(CASE WHEN status = 'delivered' THEN CAST(event_ts AS TIMESTAMP) END) as delivered_at
    FROM order_status_events
    GROUP BY order_id
),
order_revenues AS (
    SELECT
        order_id,
        SUM(amount) as net_revenue
    FROM payments
    GROUP BY order_id
)
SELECT
    o.order_id,
    DATE_TRUNC('week', CAST(o.order_ts AS TIMESTAMP))::DATE as week,
    r.cuisine,
    r.city,
    rev.net_revenue,
    date_diff('minute', ot.placed_at, ot.delivered_at) as delivery_time_minutes
FROM orders o
JOIN restaurants r ON o.restaurant_id = r.restaurant_id
LEFT JOIN order_times ot ON o.order_id = ot.order_id
LEFT JOIN order_revenues rev ON o.order_id = rev.order_id
""")

con.register('order_metrics', order_metrics_df)

# Aggregate to build the final gold reporting table
weekly_ops_report_df = q("""
SELECT
    week,
    cuisine,
    city,
    COUNT(order_id) as number_of_orders,
    ROUND(SUM(COALESCE(net_revenue, 0)), 2) as net_revenue,
    ROUND(AVG(delivery_time_minutes), 1) as avg_delivery_time_minutes
FROM order_metrics
GROUP BY week, cuisine, city
ORDER BY week, city, cuisine
""")

con.register('weekly_ops_report', weekly_ops_report_df)

print("--- Sample of the Weekly Operations Report ---")
display(weekly_ops_report_df.head(10))
print(f"Total aggregated rows: {len(weekly_ops_report_df)}")

--- Sample of the Weekly Operations Report ---


,week,cuisine,city,number_of_orders,net_revenue,avg_delivery_time_minutes
0,2026-02-23,Burgers,Birmingham,4,125.99,38.8
1,2026-02-23,Chinese,Birmingham,7,117.46,34.0
2,2026-02-23,Indian,Birmingham,1,43.99,43.0
3,2026-02-23,Italian,Birmingham,8,138.94,36.0
4,2026-02-23,Japanese,Birmingham,7,260.19,33.0
5,2026-02-23,Vegan,Birmingham,3,77.98,36.3
6,2026-02-23,Burgers,Bristol,4,66.77,40.3
7,2026-02-23,Japanese,Bristol,7,289.43,33.6
8,2026-02-23,Vegan,Bristol,2,47.55,32.0
9,2026-02-23,Burgers,Leeds,1,0.00,NaN


Total aggregated rows: 566


## 4. Check

*Show how you know your answer is right. What did you compare, count or spot check, and what did the checks show? If a check failed, say what you changed.*

Add text here as you go along. Show that you are staying in control of the build!

Already did the checking while cleaning the data

## 5. Prompts and AI notes

*Paste each prompt you used, as you sent it. Then note what you accepted, what you changed or rejected, and why. Include at least one example where the AI was wrong or needed correcting, if that happened.*

The prompt and AI noted was add throughout the process

## 6. What I could not verify

*Be honest about what remains unchecked, and what you would do next with more time.*

Need to check after the aggregatiion whether it was join correctly and that the result match with the brief

## Self-check before you stop

Tick these off in your head or in a text cell. They mirror the three things the real test is marked on.

**How was the code generated?**
- [ ] Every block of code that matters has a prompt or a source associated with it.
- [ ] I can say what each block is meant to do and what output I expected.

**Evidence of checking**
- [ ] For every table I built, I checked the number of rows against what I expected, and I can say why.
- [ ] I checked that keys I treat as unique are unique, and I looked for duplicates.
- [ ] My final table agrees with the source data in at least one total, and I show the comparison.
- [ ] I looked at individual records, not only at totals.
- [ ] I wrote down at least one thing the checks caught, and what I changed (if possible).

**Interpretation in my own words**
- [ ] I state each decision (week, which orders count, net revenue, delivery time) and why I chose it.
- [ ] I say what the figures can and cannot be trusted for.
- [ ] Section 6 lists what I did not verify, honestly.